# Cropping and saving GEBCO bathy

### Checking right BinWaves Bathy format

In [1]:
import xarray as xr
import numpy as np
import os

# Reference file path
ref_file = "/lustre/geocean/WORK/users/jen/BlueMath/methods/hybrid_downscaling/additive/BinWaves/outputs/GEBCO_NC.nc"

# Open the dataset
print(f"Reading reference file: {ref_file}")
ref_ds = xr.open_dataset(ref_file)

print("\nDetailed Reference File Structure:")
print("=" * 50)

# File existence and size
file_size = os.path.getsize(ref_file) / (1024 * 1024)  # Size in MB
print(f"File exists: {os.path.exists(ref_file)}")
print(f"File size: {file_size:.2f} MB")

# Dimensions
print("\nDimensions:")
print("-" * 20)
for dim_name, dim_size in ref_ds.dims.items():
    print(f"{dim_name}: {dim_size}")

# Coordinates
print("\nCoordinates:")
print("-" * 20)
for coord_name, coord in ref_ds.coords.items():
    print(f"Coordinate: {coord_name}")
    print(f"  Shape: {coord.shape}")
    print(f"  Dtype: {coord.dtype}")
    print(f"  Values range: [{coord.values.min()}, {coord.values.max()}]")
    print(f"  Attributes:")
    for attr_name, attr_value in coord.attrs.items():
        print(f"    {attr_name}: {attr_value}")

# Variables
print("\nVariables:")
print("-" * 20)
for var_name, var in ref_ds.data_vars.items():
    print(f"Variable: {var_name}")
    print(f"  Dimensions: {var.dims}")
    print(f"  Shape: {var.shape}")
    print(f"  Dtype: {var.dtype}")
    if var.size > 0:
        try:
            print(f"  Values range: [{np.nanmin(var.values)}, {np.nanmax(var.values)}]")
        except:
            print(f"  Values range: Cannot compute (possibly non-numeric or empty)")
    print(f"  Attributes:")
    for attr_name, attr_value in var.attrs.items():
        print(f"    {attr_name}: {attr_value}")
    print(f"  Encoding:")
    for enc_name, enc_value in var.encoding.items():
        print(f"    {enc_name}: {enc_value}")

# Global attributes
print("\nGlobal Attributes:")
print("-" * 20)
for attr_name, attr_value in ref_ds.attrs.items():
    print(f"{attr_name}: {attr_value}")

# Dataset encoding
print("\nDataset Encoding:")
print("-" * 20)
for enc_name, enc_value in ref_ds.encoding.items():
    print(f"{enc_name}: {enc_value}")

ref_ds.close()

Reading reference file: /lustre/geocean/WORK/users/jen/BlueMath/methods/hybrid_downscaling/additive/BinWaves/outputs/GEBCO_NC.nc

Detailed Reference File Structure:
File exists: True
File size: 31.07 MB

Dimensions:
--------------------
cy: 2228
cx: 1825

Coordinates:
--------------------
Coordinate: cx
  Shape: (1825,)
  Dtype: float64
  Values range: [363166.25071576727, 545566.2507157673]
  Attributes:
Coordinate: cy
  Shape: (2228,)
  Dtype: float64
  Values range: [3873132.821336043, 4095832.821336043]
  Attributes:

Variables:
--------------------
Variable: elevation
  Dimensions: ('cy', 'cx')
  Shape: (2228, 1825)
  Dtype: float64
  Values range: [-3054.5968036918935, 32.14387611228621]
  Attributes:
    standard_name: height_above_mean_sea_level
    long_name: Elevation relative to sea level
    units: m
    grid_mapping: crs
    sdn_parameter_urn: SDN:P01::BATHHGHT
    sdn_parameter_name: Sea floor height (above mean sea level) {bathymetric height}
    sdn_uom_urn: SDN:P06::UL

In [1]:
import xarray as xr
import numpy as np
from scipy.interpolate import griddata
from pyproj import Transformer
import geopandas as gpd
from shapely.geometry import box

# File paths
input_nc_path = "/lustre/geocean/WORK/users/jen/BlueMath/methods/hybrid_downscaling/additive/BinWaves/outputs_all/carolinas_gebco_utm18_cy_cx.nc"
output_nc_path = "/lustre/geocean/WORK/users/jen/BlueMath/methods/hybrid_downscaling/additive/BinWaves/outputs/GEBCO_Duke_100m.nc"

# Open the datasets
print(f"Reading input file: {input_nc_path}")
ds = xr.open_dataset(input_nc_path)

# Define your area of interest (AOI) in latitude/longitude
lon_min, lon_max = -76.4, -74.4
lat_min, lat_max = 34.8, 36.8

# Transform AOI to UTM 18N
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32618", always_xy=True)
x_min, y_min = transformer.transform(lon_min, lat_min)
x_max, y_max = transformer.transform(lon_max, lat_max)

print("Cropping dataset...")
# Crop the dataset manually
ds_cropped = ds.sel(
    cx=slice(x_min, x_max),
    cy=slice(y_min, y_max)
)

print("Creating 100m grid...")
# Create a regular grid with 100m spacing
x_new = np.arange(x_min, x_max, 100)
y_new = np.arange(y_min, y_max, 100)
X_new, Y_new = np.meshgrid(x_new, y_new)

# Identify the bathymetry variable
bathy_var_name = None
for var in ds_cropped.data_vars:
    if var not in ['cx', 'cy'] and len(ds_cropped[var].dims) == 2:
        bathy_var_name = var
        break

if bathy_var_name is None:
    raise ValueError("Cannot identify bathymetry variable in the dataset")

print(f"Identified bathymetry variable: {bathy_var_name}")

# Extract the original grid coordinates and values
cx_orig = ds_cropped.cx.values
cy_orig = ds_cropped.cy.values
CX_orig, CY_orig = np.meshgrid(cx_orig, cy_orig)
bathy_values = ds_cropped[bathy_var_name].values

# Reshape for interpolation
points = np.column_stack((CX_orig.flatten(), CY_orig.flatten()))
values = bathy_values.flatten()

# Remove NaN values
valid_indices = ~np.isnan(values)
points = points[valid_indices]
values = values[valid_indices]

print("Interpolating to 100m grid...")
# Interpolate to the new grid
bathy_interp = griddata(points, values, (X_new, Y_new), method='linear')

print("Creating new dataset with correct structure...")
# Create a new xarray dataset with the exact same structure as the reference file
ds_new = xr.Dataset(
    data_vars={
        'elevation': (['cy', 'cx'], bathy_interp)
    },
    coords={
        'cx': x_new,
        'cy': y_new
    }
)

# Add variable attributes
ds_new['elevation'].attrs = {
    'standard_name': "height_above_mean_sea_level",
    'long_name': "Elevation relative to sea level",
    'units': "m",
    'grid_mapping': "crs",
    'sdn_parameter_urn': "SDN:P01::BATHHGHT",
    'sdn_parameter_name': "Sea floor height (above mean sea level) {bathymetric height}",
    'sdn_uom_urn': "SDN:P06::ULAA",
    'sdn_uom_name': "Metres"
}

# Add only the supported encoding parameters
encoding = {
    'elevation': {
        'zlib': False,
        'shuffle': False,
        'complevel': 0,
        'fletcher32': False,
        'contiguous': True,
        'chunksizes': None,
        'dtype': 'float64',
        '_FillValue': np.nan
    }
}

print(f"Saving to {output_nc_path}...")
# Save to NetCDF with the filtered encoding
ds_new.to_netcdf(output_nc_path, encoding=encoding)
print(f"Successfully saved cropped and interpolated data to {output_nc_path} with exact same format as reference file")

Reading input file: /lustre/geocean/WORK/users/jen/BlueMath/methods/hybrid_downscaling/additive/BinWaves/outputs_all/carolinas_gebco_utm18_cy_cx.nc
Cropping dataset...
Creating 100m grid...
Identified bathymetry variable: elevation
Interpolating to 100m grid...
Creating new dataset with correct structure...
Saving to /lustre/geocean/WORK/users/jen/BlueMath/methods/hybrid_downscaling/additive/BinWaves/outputs/GEBCO_Duke_100m.nc...
Successfully saved cropped and interpolated data to /lustre/geocean/WORK/users/jen/BlueMath/methods/hybrid_downscaling/additive/BinWaves/outputs/GEBCO_Duke_100m.nc with exact same format as reference file
